In [1]:
import sys
sys.path.append("..")
import os
import torch
import json
import yaml
from pathlib import Path
import gc
from datasets import DatasetDict, load_from_disk
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training, PeftModel

from transformers import AutoTokenizer, AutoModelForCausalLM
from src.data import PROJECT_ROOT
from src.dataset_cot import prepare_direct_pythia_dataset, get_cot_prompt_template, evaluate_cot_capability
from src.llm_upgrade import (
    prepare_model_for_finetune,
    train_lora_model,
    save_finetuned_model,
    wrap_for_transformer_lens,
    create_quantization_config
)

# Параметры Progressive Fine-Tune

In [2]:
EXP_ID = "exp8-2"
MODEL_NAME = "EleutherAI/pythia-1b-deduped"
VARIANT = "depth-1"
USE_SMALL = False

In [3]:
EPOCHS = 3
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 5.0e-5
MAX_LENGTH = 512
SAVE_STEPS = 500
LOGGING_STEPS = 200
EVAL_STEPS = 200

In [4]:
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

In [5]:
TARGET_MODULES = ["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"]

In [6]:
OUTPUT_DIR = str(PROJECT_ROOT / f"results/checkpoints/finetune/{EXP_ID}")
ADAPTER_SAVE_PATH = str(PROJECT_ROOT / f"results/checkpoints/finetune/{EXP_ID}/lora_final")

In [7]:
# Точка старта: успешно обученный чекпоинт depth-0
DEPTH0_CHECKPOINT = str(PROJECT_ROOT / "results/checkpoints/finetune/exp3-2/checkpoint-6000")

In [8]:
cache_path = PROJECT_ROOT / "data" / "processed" / f"ruletaker_{VARIANT}"
raw_dataset = load_from_disk(str(cache_path))

In [9]:
# TRAIN_SIZE = 10000
DEV_SIZE = 500
SEED = 42

# Подготовка датасета

In [10]:
train_subset = raw_dataset["train"].shuffle(seed=SEED)
dev_subset = raw_dataset["dev"].shuffle(seed=SEED).select(range(min(DEV_SIZE, len(raw_dataset["dev"]))))

In [11]:
train_dataset = prepare_direct_pythia_dataset(train_subset)
dev_dataset   = prepare_direct_pythia_dataset(dev_subset)

In [12]:
len(train_dataset)

61699

In [13]:
len(dev_subset)

500

# Обучение

In [14]:
# qlora
quantization_config = create_quantization_config(use_4bit=True)

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [16]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"  # Flash Attention часто нестабилен с LoRA на Windows
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [17]:
print(f"Подключение адаптера depth-0 из: {DEPTH0_CHECKPOINT}")
model = PeftModel.from_pretrained(base_model, DEPTH0_CHECKPOINT, is_trainable=True)
model.print_trainable_parameters()

Подключение адаптера depth-0 из: C:\MyPythonProjects\mephi_diss\results\checkpoints\finetune\exp3-2\checkpoint-6000
trainable params: 8,388,608 || all params: 1,020,170,240 || trainable%: 0.8223


In [18]:
torch.cuda.empty_cache()
gc.collect()

104

In [19]:
train_config = {
    "output_dir": OUTPUT_DIR,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "max_length": MAX_LENGTH,
    "logging_steps": LOGGING_STEPS,
    "save_steps": SAVE_STEPS,
    "eval_steps": EVAL_STEPS,
    "use_wandb": False
}

In [20]:
trained_model, metrics = train_lora_model(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    config=train_config,
    max_length=MAX_LENGTH
)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
200,0.677100,0.687972
400,0.657900,0.683676
600,0.656200,0.680556
800,0.649600,0.680169
1000,0.650200,0.680341
1200,0.645900,0.679843
1400,0.637000,0.680720
1600,0.641100,0.680984
1800,0.637400,0.683228
2000,0.634300,0.683566


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /EleutherAI/pythia-1b-deduped/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 68aa3e4d-12ad-4a05-8f05-d986f79981e2)')' thrown while requesting HEAD https://huggingface.co/EleutherAI/pythia-1b-deduped/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /EleutherAI/pythia-1b-deduped/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: ce46f71f-47fb-47a3-8487-cdc4b562dbc9)')' thrown while requesting HEAD https://huggingface.co/EleutherAI/pythia-1b-deduped/resolve/main/config.json
Retrying

KeyboardInterrupt: 

In [21]:
save_finetuned_model(trained_model, tokenizer, ADAPTER_SAVE_PATH)

NameError: name 'trained_model' is not defined

# Тест дообученной модели

In [24]:
BEST_CHECKPOINT = 1000
adapter_path = PROJECT_ROOT / f"results/checkpoints/finetune/{EXP_ID}/checkpoint-{BEST_CHECKPOINT}"

In [25]:
hooked_model, _ = wrap_for_transformer_lens(
    MODEL_NAME,
    str(adapter_path),
    device="cuda"
)

Loaded pretrained model EleutherAI/pythia-1b-deduped into HookedTransformer


In [26]:
eval_prefix = "{theory} {assertion} The assertion is"

In [27]:
full_test = load_from_disk(str(cache_path))["test"]

In [28]:
test_acc = evaluate_cot_capability(
    model=hooked_model,
    dataset=full_test,
    prompt_template=eval_prefix,
    batch_size=32,
    device="cuda"
)

Evaluating CoT capability: 100%|██████████| 556/556 [1:18:20<00:00,  8.45s/it]


In [29]:
results = {
    "experiment_id": EXP_ID,
    "model": MODEL_NAME,
    "variant": VARIANT,
    "best_checkpoint": f"checkpoint-{BEST_CHECKPOINT}",
    "val_loss_at_checkpoint": 0.680341,
    "test_accuracy": float(test_acc),
    "config": {
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lora_r": LORA_R, "train_size": "full"
    }
}

In [30]:
results_path = Path(OUTPUT_DIR) / "results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

In [31]:
print(f"Experiment: {EXP_ID}")
print(f"Checkpoint: {adapter_path.name}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Results saved: {results_path}")

Experiment: exp8-2
Checkpoint: checkpoint-1000
Test Accuracy: 0.8271
Results saved: C:\MyPythonProjects\mephi_diss\results\checkpoints\finetune\exp8-2\results.json


# Диагностика предсказаний

In [32]:
test_examples = raw_dataset["test"].select(range(5))
eval_prefix = "{theory} {assertion} The assertion is"

In [33]:
with torch.no_grad():
    for i, ex in enumerate(test_examples):
        prompt = eval_prefix.format(theory=ex["theory"], assertion=ex["assertion"])
        tokens = hooked_model.to_tokens([prompt], prepend_bos=True).to("cuda")
        logits = hooked_model(tokens)

        # Индексы целевых токенов
        true_id = hooked_model.tokenizer.encode("True", add_special_tokens=False)[-1]
        false_id = hooked_model.tokenizer.encode("False", add_special_tokens=False)[-1]

        # Длина промпта (последний токен находится по этому индексу после добавления BOS)
        prompt_len = len(hooked_model.tokenizer.encode(prompt, add_special_tokens=False))

        logit_t = logits[0, prompt_len, true_id].item()
        logit_f = logits[0, prompt_len, false_id].item()
        pred = "True" if logit_t > logit_f else "False"
        true_label = "True" if ex["label"] else "False"

        print(f"[{i+1}] Предсказание: {pred:5s} | Логиты T/F: {logit_t:+.2f} / {logit_f:+.2f} | Истина: {true_label} | Совпало: {pred == true_label}")

[1] Предсказание: True  | Логиты T/F: +15.50 / +10.62 | Истина: True | Совпало: True
[2] Предсказание: False | Логиты T/F: +10.06 / +12.75 | Истина: False | Совпало: True
[3] Предсказание: True  | Логиты T/F: +15.06 / +13.12 | Истина: True | Совпало: True
[4] Предсказание: False | Логиты T/F: +13.81 / +15.06 | Истина: False | Совпало: True
[5] Предсказание: False | Логиты T/F: +14.44 / +15.38 | Истина: True | Совпало: False


In [34]:
del hooked_model
torch.cuda.empty_cache()